# M4U3 — Roboflow step 1: split images into upload batches

Sorts the captured frames into `train` / `val` / `test` folders so each batch can be
uploaded to Roboflow with the correct split assigned in one go.

**Run this locally**, where the image files are — not in Colab.

Splits are assigned **by room**, not at random. The whole dataset comes from one
building, so a random split would put near-identical frames of the same wall on
both sides and produce validation scores that measure memorisation.

| split | rooms | expected frames |
|---|---|---|
| train | closet | 137 |
| val | kitchen, hallway | 44 |
| test | living, office, bedroom | 46 |

Excluded from upload: `exclude == "yes"` (5 frames) and `split == "holdout"` (12 frames).
**The holdout frames must never enter the Roboflow project.**


## 1. Configuration

`SRC` is searched recursively, so the folder layout underneath it doesn't matter —
matching is on `photo_id` + any image extension.

If the images are in Google Drive and you'd rather run this in Colab, mount the drive
and point `SRC` at the mounted path:

```python
from google.colab import drive; drive.mount('/content/drive')
SRC = Path('/content/drive/MyDrive/storm-damage/00_raw')
```


In [3]:
%pip install pandas
from pathlib import Path
import pandas as pd, shutil

LOG = Path("areas.csv")
SRC = Path("00_raw")                    # unchanged - recursive, nesting is fine
DST = Path("01_working/upload")         # was 01_upload

EXTS = {".jpg", ".jpeg", ".png", ".heic", ".JPG", ".JPEG", ".PNG", ".HEIC"}
EXPECTED = {"train": 137, "val": 44, "test": 46}

print("log :", LOG.resolve())
print("src :", SRC.resolve())
print("dst :", DST.resolve())


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 9.8/9.8 MB 63.1 MB/s  0:00:00

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
 

## 2. Load the log and check it is internally consistent

In [4]:
df = pd.read_csv(LOG, sep=";", encoding="utf-8-sig", dtype=str).fillna("")
print(f"{len(df)} rows, {len(df.columns)} columns")

# room -> split must be one-to-one, or the split is not actually by room
bad = df[df.split != "holdout"].groupby("room").split.nunique()
bad = bad[bad > 1]
assert bad.empty, f"rooms split inconsistently: {list(bad.index)}"

print()
print(pd.crosstab(df.room, df.split).to_string())

244 rows, 14 columns

split    holdout  test  train  val
room                              
bedroom        0     3      0    0
closet         0     0    142    0
hallway        5     0      0   10
kitchen        1     0      0   34
living         6    31      0    0
office         0    12      0    0


## 3. Build the manifest

One row per file to copy. `exclude=yes` and `holdout` rows are dropped here, which is
the only place that decision is made.


In [5]:
work = df[(df.exclude != "yes") & (df.split.isin(["train", "val", "test"]))].copy()
print(f"{len(work)} frames selected for upload")
print(work.split.value_counts().reindex(["train", "val", "test"]).to_string())

for split, n in EXPECTED.items():
    got = int((work.split == split).sum())
    flag = "ok" if got == n else "MISMATCH"
    print(f"  {split:<6} {got:>4} / {n:<4} {flag}")

227 frames selected for upload
split
train    137
val       44
test      46
  train   137 / 137  ok
  val      44 / 44   ok
  test     46 / 46   ok


## 4. Dry run — resolve every filename before copying anything

Anything listed as **missing** or **ambiguous** is a disagreement between the log and
the disk. Resolve those before proceeding: a missing file means the log references
something that isn't there, an ambiguous one means two files share a stem.


In [7]:
import re

def num(stem):
    m = re.search(r"(\d+)$", stem)
    return m.group(1) if m else None

index = {}
for p in SRC.rglob("*"):
    if p.is_file() and p.suffix in EXTS:
        n = num(p.stem)
        if n:
            index.setdefault(n, []).append(p)

missing, ambiguous, resolved = [], [], []
for _, r in work.iterrows():
    n = num(r.photo_id)
    hits = index.get(n, [])
    if len(hits) == 0:
        missing.append((r.photo_id, n))
    elif len(hits) > 1:
        ambiguous.append((r.photo_id, [str(h) for h in hits]))
    else:
        resolved.append((hits[0], r.split, r.photo_id))   # src, split, new stem

print(f"resolved  : {len(resolved)}")
print(f"missing   : {len(missing)}")
print(f"ambiguous : {len(ambiguous)}")

for pid, n in missing:
    print("  MISSING  ", pid, "(no file numbered", n, ")")
for pid, paths in ambiguous:
    print("  AMBIGUOUS", pid, paths)

logged_nums = {num(p) for p in df.photo_id}
orphans = sorted(k for k in index if k not in logged_nums)
print(f"\non disk but not in the log ({len(orphans)}): {orphans}")

resolved  : 227
missing   : 0
ambiguous : 0

on disk but not in the log (5): ['6948', '7089', '7090', '7091', '7092']


## 5. Copy

Set `DRY_RUN = False` only once section 4 reports zero missing and zero ambiguous.

Files are **copied**, not moved — `00_raw` stays untouched.


In [8]:
DRY_RUN = False

if DRY_RUN:
    print("DRY RUN — nothing written.")
    for split in ["train", "val", "test"]:
        n = sum(1 for _, s, _ in resolved if s == split)
        print(f"  would write {n:>4} files to {DST / split}")
    print("\nexample rename:")
    for src_path, split, stem in resolved[:3]:
        print(f"  {src_path.name}  ->  {split}/{stem}{src_path.suffix}")
else:
    for split in ["train", "val", "test"]:
        (DST / split).mkdir(parents=True, exist_ok=True)
    for src_path, split, stem in resolved:
        shutil.copy2(src_path, DST / split / f"{stem}{src_path.suffix}")
    print(f"copied and renamed {len(resolved)} files")

copied and renamed 227 files


## 6. Verify what actually landed

In [9]:
ok = True
for split, n in EXPECTED.items():
    d = DST / split
    got = len([p for p in d.glob("*") if p.suffix in EXTS]) if d.exists() else 0
    status = "ok" if got == n else "MISMATCH"
    if got != n:
        ok = False
    print(f"{split:<6} {got:>4} / {n:<4} {status}")

print()
print("ready to upload" if ok else "resolve the mismatches before uploading")

train   137 / 137  ok
val      44 / 44   ok
test     46 / 46   ok

ready to upload


## 7. Holdout — kept deliberately separate

These 12 frames are the "new images" for the inference evidence pack. They cover wall
regions never photographed for the dataset: `LIVING-F` and `KITCHEN-E` are new walls,
`HALLWAY-B1` and `HALLWAY-C1` are lower sections near the skirting of walls whose upper
sections are in `val`.

They are copied to a folder of their own so they cannot be dragged into a Roboflow
upload by accident. **Do not upload this folder.**


In [ ]:
"""
HOLD_DST = Path("02_holdout")

hold = df[df.split == "holdout"]
print(f"{len(hold)} holdout frames")
print(hold.groupby("area_id").size().to_string())

if not DRY_RUN:
    HOLD_DST.mkdir(parents=True, exist_ok=True)
    n = 0
    for _, r in hold.iterrows():
        hits = index.get(r.photo_id, [])
        if len(hits) == 1:
            shutil.copy2(hits[0], HOLD_DST / hits[0].name)
            n += 1
        else:
            print("  unresolved:", r.photo_id, hits)
    print(f"\ncopied {n} holdout frames to {HOLD_DST}")
else:
    print("\nDRY RUN — holdout not copied.")
"""

In [ ]:
print(len([p for p in Path("99_holdout").glob("*") if p.suffix in EXTS]))   # expect 12

---

## Next

1. Confirm section 6 reports **137 / 44 / 46**.
2. In Roboflow: create the project (Object Detection), define the five classes
   (`blistering`, `crack`, `damp_patch`, `mould`, `peeling_paint`) **before** uploading.
3. Upload `01_upload/train`, `01_upload/val` and `01_upload/test` as three separate
   batches, setting the split in the upload dialog each time.
4. Decline "add similar images".
